# Jet thrust with QAOA

This tutorial turns a tiny, momentum-balanced $e^+e^-\to$ hadrons event into the thrust QUBO used in quantum jet-clustering studies, solves it with QARP's QAOA, and checks it against exhaustive enumeration.

**Why this event is deliberately easy.** The two momentum lobes below are arranged mainly along the $x$ axis, so we can verify the answer by inspection: particles with $p_x>0$ form one hemisphere and particles with $p_x<0$ form the other. This obvious left/right split is an intentional sanity check for the encoding and QAOA result. It is not the general solution to the thrust problem: for a generic event, the optimal thrust axis is not known in advance and need not align with any coordinate axis. Hemisphere membership is then determined by the sign of $\vec p_i\cdot\hat n_{\mathrm{thrust}}$, after optimizing the axis.

For a binary assignment $x_i\in\{0,1\}$, define

$$
P(x)=\sum_i x_i\,\vec p_i,\qquad
T(x)=\frac{2\lVert P(x)\rVert}{\sum_i\lVert\vec p_i\rVert}.
$$

When the event has zero total momentum, maximizing $T^2$ is equivalent to maximizing $x^TQx$ with $Q_{ij}=\vec p_i\cdot\vec p_j$. We therefore minimize $E(x)=-x^TQx$.

**Encoding and scope.** There is one qubit per particle. The momenta become coefficients of an Ising Hamiltonian; they are **not** amplitude-loaded into a $2^n$-entry quantum state. Exhaustive enumeration is retained only as a small-instance validation oracle. This is a transparent algorithm demonstration, not a claim of quantum advantage.

References: the [HEP quantum-computing review](https://arxiv.org/abs/2307.03236), [Quantum Algorithms for Jet Clustering](https://arxiv.org/abs/1908.08949), and [Quantum Annealing for Jet Clustering with Thrust](https://arxiv.org/abs/2205.02814).

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np

import qarp
from qarp.algorithms import QAOA, Sampler
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator
from qarp.optimizers import ScipyOptimizer
from qarp.resources import Stage, estimate

np.set_printoptions(precision=4, suppress=True)

## 1. A toy event

The six three-momenta form two visibly back-to-back lobes, with small transverse components. Their sum vanishes up to floating-point precision.

In [ ]:
momenta = np.array(
    [
        [0.95, 0.10, 0.05],
        [0.65, -0.08, 0.02],
        [0.35, 0.06, -0.04],
        [-0.90, -0.12, -0.06],
        [-0.60, 0.09, 0.03],
        [-0.45, -0.05, 0.00],
    ]
)
n_particles = len(momenta)
assert np.allclose(momenta.sum(axis=0), 0.0)

fig, ax = plt.subplots(figsize=(12, 12))

for i, (px, py, _) in enumerate(momenta):
    ax.arrow(0, 0, px, py, head_width=0.035, length_includes_head=True)
    ax.text(1.08 * px, 1.08 * py, f"$p_{i}$")
ax.axhline(0, color="0.85", linewidth=1)
ax.axvline(0, color="0.85", linewidth=1)
ax.set(xlabel="$p_x$", ylabel="$p_y$", title="Toy event (transverse projection)")
ax.set_aspect("equal")
plt.show()

## 2. Build and validate the QUBO formulation

Expanding $-x^TQx$ gives linear terms $a_i=-Q_{ii}$ and pair terms $b_{ij}=-2Q_{ij}$. The helper maps $x_i=(1-Z_i)/2$ into a diagonal QARP `QubitOperator`. The assertion checks every diagonal element against the original QUBO before any variational optimization is attempted.

In [ ]:
def qubo_to_ising(linear, quadratic):
    """Map E(x)=sum a_i x_i + sum b_ij x_i x_j to x_i=(1-Z_i)/2."""
    n_qubits = len(linear)
    constant = 0.5 * sum(linear.values()) + 0.25 * sum(quadratic.values())
    fields = {i: -0.5 * linear[i] for i in range(n_qubits)}
    for (i, j), value in quadratic.items():
        fields[i] -= 0.25 * value
        fields[j] -= 0.25 * value

    hamiltonian = QubitOperator((), constant)
    for i, value in fields.items():
        if abs(value) > 1e-12:
            hamiltonian += QubitOperator(f"Z{i}", value)
    for (i, j), value in quadratic.items():
        if abs(value) > 1e-12:
            hamiltonian += QubitOperator(f"Z{i} Z{j}", 0.25 * value)
    return hamiltonian

In [ ]:
gram = momenta @ momenta.T
linear = {i: -gram[i, i] for i in range(n_particles)}
quadratic = {
    (i, j): -2.0 * gram[i, j] for i in range(n_particles) for j in range(i + 1, n_particles)
}
cost_hamiltonian = qubo_to_ising(linear, quadratic)


def qubo_energy(bits):
    x = np.asarray(bits, dtype=float)
    return -float(x @ gram @ x)


def thrust(bits):
    selected_momentum = np.asarray(bits, dtype=float) @ momenta
    return 2.0 * np.linalg.norm(selected_momentum) / np.linalg.norm(momenta, axis=1).sum()


ising_diagonal = cost_hamiltonian.sparse_matrix(n_particles).diagonal().real
for bits in itertools.product([0, 1], repeat=n_particles):
    basis_index = sum(bit << qubit for qubit, bit in enumerate(bits))
    assert np.isclose(ising_diagonal[basis_index], qubo_energy(bits))

classical_solutions = sorted(
    (qubo_energy(bits), thrust(bits), bits)
    for bits in itertools.product([0, 1], repeat=n_particles)
)
ground_energy = classical_solutions[0][0]
ground_states = {
    bits for energy, _, bits in classical_solutions if np.isclose(energy, ground_energy)
}

print("Ground-state partitions:", ground_states)
print(f"Maximum thrust: {classical_solutions[0][1]:.6f}")
assert ground_states == {(1, 1, 1, 0, 0, 0), (0, 0, 0, 1, 1, 1)}

A measured bitstring is $(x_0,…,x_5)$ in `q0 first` order, with one bit per input particle $p_i$. Here $x_i=1$ assigns $p_i$ to the selected jet hemisphere and $x_i=0$ assigns it to the opposite hemisphere. Thus `(1, 1, 1, 0, 0, 0)` groups $p_0,p_1,p_2$ together and $p_3,p_4,p_5$ together. Its complement describes exactly the same *unlabelled* two-jet partition, so the twofold degeneracy is physical rather than an error.

## 3. Optimize a depth-2 QAOA circuit

QARP constructs the alternating cost and mixer layers, evaluates the exact expectation value with its state-vector engine, and passes the four angles to SciPy. Exact probabilities keep this tutorial deterministic; changing `qarp.EXACT` to a shot count demonstrates finite-sampling noise.

In [ ]:
optimizer = ScipyOptimizer(
    "Nelder-Mead",
    options={"maxiter": 300, "xatol": 1e-5, "fatol": 1e-7},
)
qaoa = QAOA(
    cost_hamiltonian,
    n_layers=2,
    initial_parameters=np.array([0.35, 0.80, 0.65, 0.25]),
    optimizer=optimizer,
    save_energy_history=True,
).build()

minimum_expectation, optimal_parameters = qaoa.run()
print(f"Final expectation value: {minimum_expectation:.6f}")
print("Optimal parameters:", optimal_parameters)

In [ ]:
final_state = qaoa.get_final_state_block()
sampler = Sampler(ket=final_state, n_shots=qarp.EXACT)
engine = QarpEngine(n_shots=qarp.EXACT)
engine.build([sampler])
probabilities = engine.run()[0]
ranked = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)

success_probability = sum(
    probability for bits, probability in probabilities.items() if bits in ground_states
)
most_likely_bits = ranked[0][0]
print("Most likely partition (q0 first):", most_likely_bits)
print(f"Ground-space probability: {success_probability:.3f}")
print(f"Thrust of most likely partition: {thrust(most_likely_bits):.6f}")

assert most_likely_bits in ground_states
assert success_probability > 0.60

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].plot(qaoa.energy_history)
axes[0].axhline(ground_energy, color="black", linestyle="--", label="exact ground energy")
axes[0].set(
    xlabel="objective evaluation", ylabel=r"$\langle H_C\rangle$", title="Classical optimization"
)
axes[0].legend()

top = ranked[:10]
labels = ["".join(map(str, bits)) for bits, _ in top]
axes[1].bar(labels, [probability for _, probability in top])
axes[1].set(
    xlabel="bit string ($q_0$ first)", ylabel="probability", title="Largest QAOA probabilities"
)
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
## Circuit visualization of the QAOA state
final_state = qaoa.get_final_state_block()
#final_state.plot(decompose_boxes=True)
final_state.plot()

## 4. Extension: increase the QAOA depth

The preceding history shows convergence of the classical optimizer **at fixed depth** $p=2$; it does not imply that the variational state has reached the exact ground state. To separate those ideas, we now train $p=2,3,4$ layerwise. When a layer is added, the optimized $\beta$ and $\gamma$ sequences are linearly interpolated to initialize the deeper circuit. This continuation strategy is deterministic and avoids restarting every depth from unrelated angles.

In [ ]:
def interpolate_qaoa_parameters(parameters):
    """Lift [beta_0...beta_p-1, gamma_0...gamma_p-1] from p to p + 1."""
    parameters = np.asarray(parameters, dtype=float)
    depth = len(parameters) // 2

    def interpolate(sequence):
        extended = np.empty(depth + 1)
        extended[0] = sequence[0]
        extended[-1] = sequence[-1]
        for index in range(1, depth):
            weight = index / depth
            extended[index] = weight * sequence[index - 1] + (1.0 - weight) * sequence[index]
        return extended

    return np.concatenate([interpolate(parameters[:depth]), interpolate(parameters[depth:])])


def exact_qaoa_distribution(model):
    depth_sampler = Sampler(ket=model.get_final_state_block(), n_shots=qarp.EXACT)
    depth_engine = QarpEngine(n_shots=qarp.EXACT)
    depth_engine.build([depth_sampler])
    return depth_engine.run()[0]


qaoa_by_depth = {2: qaoa}
energy_by_depth = {2: float(minimum_expectation)}
history_by_depth = {2: np.asarray(qaoa.energy_history)}
probability_by_depth = {2: probabilities}
ground_probability_by_depth = {2: success_probability}
layerwise_parameters = np.asarray(optimal_parameters)

for depth in (3, 4):
    layerwise_parameters = interpolate_qaoa_parameters(layerwise_parameters)
    depth_optimizer = ScipyOptimizer(
        "Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-7, "fatol": 1e-10},
    )
    depth_qaoa = QAOA(
        cost_hamiltonian,
        n_layers=depth,
        initial_parameters=layerwise_parameters,
        optimizer=depth_optimizer,
        save_energy_history=True,
    ).build()
    depth_energy, layerwise_parameters = depth_qaoa.run()
    depth_probabilities = exact_qaoa_distribution(depth_qaoa)

    qaoa_by_depth[depth] = depth_qaoa
    energy_by_depth[depth] = float(depth_energy)
    history_by_depth[depth] = np.asarray(depth_qaoa.energy_history)
    probability_by_depth[depth] = depth_probabilities
    ground_probability_by_depth[depth] = sum(
        probability for bits, probability in depth_probabilities.items() if bits in ground_states
    )

depths = np.array(sorted(qaoa_by_depth))
for depth in depths:
    gap = energy_by_depth[depth] - ground_energy
    probability = ground_probability_by_depth[depth]
    print(
        f"p={depth}: energy={energy_by_depth[depth]:.6f}, "
        f"gap={gap:.6f}, ground-space probability={probability:.3f}"
    )

assert energy_by_depth[4] - ground_energy < 0.08
assert ground_probability_by_depth[4] > 0.96

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7), sharey=True)
for axis, depth in zip(axes, depths, strict=True):
    axis.plot(history_by_depth[depth], color=f"C{depth - 2}")
    axis.axhline(ground_energy, color="black", linestyle="--", label="exact ground energy")
    axis.set(
        xlabel="objective evaluation",
        title=f"Fixed-depth optimization: p={depth}",
    )
axes[0].set_ylabel(r"$\langle H_C\rangle$")
axes[0].legend()
plt.tight_layout()
plt.show()

final_energies = np.array([energy_by_depth[depth] for depth in depths])
ground_probabilities = np.array([ground_probability_by_depth[depth] for depth in depths])
uniform_ground_probability = len(ground_states) / 2**n_particles

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(depths, final_energies, marker="o", label="optimized QAOA")
axes[0].axhline(ground_energy, color="black", linestyle="--", label="exact ground energy")
axes[0].set(
    xlabel="QAOA depth p",
    ylabel=r"optimized $\langle H_C\rangle$",
    xticks=depths,
    title="Energy improves with depth",
)
axes[0].legend()

axes[1].plot(depths, ground_probabilities, marker="o", color="tab:green")
axes[1].axhline(
    uniform_ground_probability,
    color="0.4",
    linestyle=":",
    label="uniform baseline",
)
axes[1].set(
    xlabel="QAOA depth p",
    ylabel="ground-space probability",
    xticks=depths,
    ylim=(0.0, 1.02),
    title="Correct-partition sampling probability",
)
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Read the physics result and count resources

The most likely $p=2$ bit string already selects the correct jet hemisphere; its complement represents the other labelling of the same two-jet event. The string is a sampled classical assignment, not the optimized expectation value itself: QAOA produces a probability distribution over all assignments. Increasing $p$ primarily concentrates more probability in that two-dimensional ground space. The resource counts below expose the corresponding circuit-depth cost.

In [ ]:
colors = ["tab:blue" if bit else "tab:orange" for bit in most_likely_bits]
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, ((px, py, _), color) in enumerate(zip(momenta, colors, strict=True)):
    ax.arrow(0, 0, px, py, color=color, head_width=0.035, length_includes_head=True)
    ax.text(1.08 * px, 1.08 * py, f"$p_{i}$", color=color)
ax.axhline(0, color="0.85", linewidth=1)
ax.axvline(0, color="0.85", linewidth=1)
ax.set(xlabel="$p_x$", ylabel="$p_y$", title="QAOA hemisphere assignment")
ax.set_aspect("equal")
plt.show()

for depth in depths:
    logical = estimate(qaoa_by_depth[depth].get_final_state_block())[Stage.LOGICAL]
    print(
        f"p={depth}: {logical.n_qubits} qubits, logical depth {logical.depth}, "
        f"{logical.n_2q} two-qubit gates"
    )

## Takeaways

- The feature vector is encoded in Hamiltonian coefficients, so preparing the input does not require a generic $2^n$-amplitude loader.
- The quantum register still grows with the number of particles, and the dense thrust objective produces all-to-all $ZZ$ couplings.
- Optimizer convergence at fixed $p$ does not mean convergence to the exact ground energy. In this example, layerwise training from $p=2$ through $p=4$ closes most of the energy gap and raises the ground-space probability, while increasing circuit depth and two-qubit-gate count.
- QAOA is approximate: depth, optimization, sampling, and hardware connectivity determine solution quality. The exhaustive oracle used here scales as $2^n$ and is only for tutorial validation.